# Блок 6. Практика: Визуализация и дашборды

В этом ноутбуке вы:
- Подготовите SQL-запросы для панелей Grafana
- Проверите их работоспособность
- Проанализируете результаты

**Основная работа с Grafana выполняется в веб-интерфейсе (http://localhost:3000)**

In [ ]:
import duckdb

# Используем данные из data/
PARQUET_PATH = "../data/auth_events.parquet"

# Проверяем доступность данных
duckdb.sql(f"FROM '{PARQUET_PATH}' SELECT COUNT(*) as total").show()

## 1. Подготовка запросов для дашборда

Создайте SQL-запросы для каждой панели дашборда.

**Макросы Grafana:**
- `$__timeFilter(column)` — автоматический фильтр по временному диапазону
- `$__timeGroup(column, '1h')` — группировка по времени
- `$username` — переменная для фильтрации по пользователю

In [ ]:
# TODO: Запрос для панели Time Series
# Цель: Показать количество успешных и неудачных входов по времени
# Указания: Используйте $__timeGroup и $__timeFilter
# Подсказка: GROUP BY 1 (где 1 — это первый столбец в SELECT)
duckdb.sql(f"""
    FROM '{PARQUET_PATH}'
    SELECT 
        $__timeGroup(timestamp, '1h') as time,
        SUM(CASE WHEN success THEN 1 ELSE 0 END) as successful,
        SUM(CASE WHEN NOT success THEN 1 ELSE 0 END) as failed
    WHERE $__timeFilter(timestamp)
    GROUP BY 1
    ORDER BY time
""").show()

In [ ]:
# TODO: Запрос для панели Stat (общее количество неудачных попыток)
# Цель: Показать общее количество неудачных попыток за выбранный период
# Указания: Используйте WHERE для фильтрации по success = false и временному диапазону
duckdb.sql(f"""
    FROM '{PARQUET_PATH}'
    SELECT COUNT(*) as failed_total
    WHERE success = false
      AND $__timeFilter(timestamp)
""").show()

In [ ]:
# TODO: Запрос для панели Bar Chart
# Цель: Показать TOP-10 IP по количеству неудачных попыток
# Указания: GROUP BY source_ip, ORDER BY количество по убыванию, LIMIT 10
duckdb.sql(f"""
    FROM '{PARQUET_PATH}'
    SELECT source_ip, COUNT(*) as failed_count
    WHERE success = false
      AND $__timeFilter(timestamp)
    GROUP BY source_ip
    ORDER BY failed_count DESC
    LIMIT 10
""").show()

In [ ]:
# TODO: Запрос для панели Table
# Цель: Показать последние 20 подозрительных событий (неудачные входы)
# Указания: ORDER BY timestamp по убыванию, LIMIT 20
duckdb.sql(f"""
    FROM '{PARQUET_PATH}'
    SELECT timestamp, username, source_ip, event_type
    WHERE success = false
      AND $__timeFilter(timestamp)
    ORDER BY timestamp DESC
    LIMIT 20
""").show()

In [ ]:
# TODO: Запрос для переменной $username
# Цель: Получить список всех пользователей для фильтрации
# Указания: Используйте DISTINCT
duckdb.sql(f"""
    SELECT DISTINCT username
    FROM '{PARQUET_PATH}'
    ORDER BY username
""").show()

## 2. Проверка результатов

Убедитесь, что запросы возвращают ожидаемые данные и их структура подходит для визуализации в Grafana.

**Важно:** В Grafana замените макросы `$__timeFilter` и `$__timeGroup` на их реальные значения для тестирования. Например:
- `$__timeFilter(timestamp)` → `timestamp > NOW() - INTERVAL '24 hours'`
- `$__timeGroup(timestamp, '1h')` → `DATE_TRUNC('hour', timestamp)`

## 3. Аналитика с помощью визуализации

После настройки дашборда в Grafana, ответьте на следующие вопросы:

1. Есть ли на графике времени какие-либо аномальные пики активности? Когда они происходят?
2. Какие IP-адреса наиболее активны в попытках неудачного входа?
3. Можно ли выявить какие-либо закономерности во времени, когда происходят неудачные входы?
4. Как изменилась картина, когда вы применили фильтр по конкретному пользователю (например, `dev_sergey`)?
5. Какую дополнительную информацию вы смогли бы добавить на дашборд для улучшения аналитики?

In [ ]:
# TODO: Запишите свои наблюдения и выводы по каждому вопросу

## 4. Следующие шаги

1. Убедитесь, что в директории `.env` файл настроен с правильными переменными (DB_USER, DB_PASSWORD и т.д.)
2. Запустите инфраструктуру: `docker compose up -d`
3. Дождитесь, пока все сервисы (postgres, grafana, vector) станут здоровыми
4. Откройте Grafana в браузере: http://localhost:3000
5. Войдите под учетной записью (по умолчанию: admin / admin)
6. Перейдите в `Configuration` → `Data Sources` и найдите PostgreSQL — он должен быть уже настроен через provisioning
7. Перейдите в `Dashboards` → `Browse` и найдите дашборд `Security Overview Dashboard` в папке `Security` — он должен быть уже доступен
8. Изучите готовый дашборд и убедитесь, что все панели отображают данные
9. Измените временной диапазон и убедитесь, что данные обновляются
10. Попробуйте использовать переменную `$username` для фильтрации по разным пользователям

**Поздравляем!** Вы настроили автоматическую визуализацию данных безопасности с помощью Grafana.

Помните, что хороший дашборд — это не просто красивые графики, а инструмент, который помогает быстро принимать решения. Продолжайте экспериментировать и улучшать свою аналитику.